# Chapter1 Section 1A

**Generated by** `splc --lang python/linalg`
**Domain library:** `linalg_graph.py`
**DODA note:** The source `.spl` file is the logical view; this notebook is the physical artifact for the `python/linalg` domain target.

## Inputs

| Parameter | Type | Description |
|-----------|------|-------------|
| `@concept` | `TEXT` | — |

In [ ]:
# ── python/linalg target setup — generated by splc ───────────────────────────
import sys, os, json, time
from pathlib import Path

# Locate linalg_graph.py (cookbook/71_linalg_micro_textbook or current dir)
_SEARCH = [
    Path(__file__).parent if "__file__" in dir() else Path("."),
    Path(os.environ.get("LINALG_GRAPH_DIR", ".")),
    Path("."),
]
for _p in _SEARCH:
    _p = _p.resolve()
    if (_p / "linalg_graph.py").exists():
        sys.path.insert(0, str(_p))
        break
import linalg_graph as lg
from linalg_graph import (
    build, acyclic, reducible, minimal, ancestors, restrict,
    productivity_order, in_graph, applications_of, new_primitives,
    first_radical_primitives, both_radical_primitives,
    concept_names, primitive_names,
    gap, learning_path,
)
from style_profiles import style_instruction, get_style_profile, available_styles

# ── SPL runtime config lookup — env var, then ~/.spl/config, then default
# (same precedence `spl3 configure` documents for "SPL runtime defaults";
# the notebook is a standalone artifact so it re-reads the dotenv-format
# file directly rather than importing the spl3 CLI) ──────────────────────────
def _spl_config(key: str, default: str) -> str:
    if key in os.environ:
        return os.environ[key]
    _cfg = Path.home() / ".spl" / "config"
    if _cfg.exists():
        for _line in _cfg.read_text(encoding="utf-8").splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _, _v = _line.partition("=")
                if _k.strip() == key:
                    return _v.strip().strip('"').strip("'")
    return default

# ── LLM helper (configure SPL_MODEL / SPL_LLM_TIMEOUT / SPL_OLLAMA_URL, or
# replace with your adapter) — calls Ollama's OpenAI-compatible HTTP endpoint
# directly (stream=False), the same way spl/adapters/ollama.py's OllamaAdapter
# does. Earlier this shelled out to the interactive `ollama run <model>` CLI —
# but that CLI streams a live "Thinking..." status line to stdout using raw
# ANSI cursor-control escapes (`\x1b[9D\x1b[K`, ...) to overwrite it in place,
# and `subprocess.run(capture_output=True)` captures those escapes verbatim
# into the "response", contaminating every GENERATE result, the cache, and the
# committed textbook with ~2000 stray control-character sequences. The HTTP
# API talks to the same Ollama daemon but returns clean structured JSON. ──────
def _llm_call(prompt: str, model: str | None = None) -> str:
    import urllib.request
    _m = model or os.environ.get("SPL_MODEL", "llama3.2")
    # Local models (esp. 8B+ on consumer GPUs) can take well over a minute per
    # call, and refine-loop prompts re-send the full prior section as context —
    # default generously and let SPL_LLM_TIMEOUT override per deployment.
    _timeout = int(os.environ.get("SPL_LLM_TIMEOUT", "600"))
    _url = os.environ.get("SPL_OLLAMA_URL", "http://localhost:11434") + "/v1/chat/completions"
    _payload = json.dumps({
        "model": _m,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }).encode("utf-8")
    _req = urllib.request.Request(
        _url, data=_payload, method="POST",
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(_req, timeout=_timeout) as _resp:
        _data = json.loads(_resp.read().decode("utf-8"))
    return _data["choices"][0]["message"]["content"].strip()

# ── Layer 2 content cache (write-once, content-addressed — same backend and
# semantics as spl/stdlib.py's cache_get/cache_put @spl_tools; reimplemented
# here directly against spl3.cache because this notebook is a standalone
# artifact that calls the cache without going through the SPL executor) ──────
def cache_get(concept: str, rubric_version: str = "v1") -> str:
    """Returns cached content on a hit, or the sentinel string "miss"."""
    try:
        from spl3.cache import get_content_cache
        entry = get_content_cache().get(
            concept=concept, params={}, rubric_version=rubric_version, dep_hashes={},
        )
        return entry.content if entry is not None else "miss"
    except Exception:
        return "miss"

def cache_put(concept: str, content: str, rubric_version: str = "v1") -> str:
    """Stores generated+verified content (write-once, immutable); returns the cache key."""
    try:
        from spl3.cache import get_content_cache
        entry = get_content_cache().put(
            concept=concept, content=content, provenance="machine_generated",
            params={}, rubric_version=rubric_version, dep_hashes={},
        )
        return entry.key
    except Exception as exc:
        return f"cache_put error: {exc}"

# ── Math verifier helpers ─────────────────────────────────────────────────────
def _verify_math(section: str) -> str:
    """Returns 'pass' or a description of the failure."""
    try:
        import sympy  # noqa: F401 — presence check
        return "pass"   # TODO: parse worked examples from section and recompute
    except Exception as exc:
        return f"fail: {exc}"

def _shape_check(section: str) -> str:
    """Check matrix dimension compatibility. Returns 'pass' or error."""
    return "pass"  # TODO: parse matrix expressions and check shapes

# ── Timing helpers ────────────────────────────────────────────────────────────
# Bare-name wrappers around time.time() — the SPL SOLVE-template parser does
# not yet support dotted-attribute continuations (module.attr(...)), so the
# .spl source calls these instead of `time.time()` directly. Logged per
# section and for the whole run so executions can be profiled and compared
# (e.g. across distributed workers).
def _now() -> float:
    return time.time()

def _elapsed(start: float) -> float:
    return time.time() - start

# ── Workflow parameters (override before running or use papermill) ────────────
concept = '1A_Rn_and_Cn'  # TEXT
section = ""

# Pre-build the linalg concept graph (reused by all SOLVE/ASSERT cells)
graph = lg.build()
primitives = lg.both_radical_primitives()
print(f"linalg_graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [ ]:
# ── Prompt templates (mirrors CREATE FUNCTION blocks in .spl) ────────────

WRITE_SECTION_PROMPT = """\
You are an expert linear algebra author writing one self-contained section of
a micro-textbook that follows Sheldon Axler's "Linear Algebra Done Right" (4e).

{book_context}

Write a 250-350 word section that:
1. States the key definitions precisely, using LaTeX for all mathematics
   (e.g. $\\mathbf{{F}}$, $\\mathbb{{R}}^n$, $a + bi$).
2. Includes exactly one short worked example illustrating the central idea.
3. Ends with a single sentence on why this material matters for what follows.

Output only the section text — no chapter/section numbers, no preamble, no
closing commentary.
"""

REFINE_SECTION_PROMPT = """\
Revise the section below so that it satisfies the feedback exactly. Keep every
correct sentence unchanged; touch only what the feedback flags as missing.

SECTION:
{section}

FEEDBACK:
{feedback}

Output the revised section only. No preamble. No commentary.
"""

In [ ]:
# SPL: CALL now_ms() INTO @t_workflow_start
t_workflow_start = now_ms()
print(f'@t_workflow_start:', t_workflow_start)

In [ ]:
# SPL: CALL section_context(@concept) INTO @book_context
book_context = section_context(concept)
print(f'@book_context:', book_context)

In [ ]:
# SPL: CALL section_terms(@concept) INTO @required_terms
required_terms = section_terms(concept)
print(f'@required_terms:', required_terms)

In [ ]:
# SPL: CALL now_ms() INTO @t_section_start
t_section_start = now_ms()
print(f'@t_section_start:', t_section_start)

In [ ]:
# SPL: CALL cache_get(@concept) INTO @section
section = cache_get(concept)
print(f'@section:', section)

In [ ]:
# SPL: EVALUATE @section
if section == 'miss':
    # SPL: LOGGING f'cache MISS for {@concept} — generating with the LLM' LEVEL INFO
    print('[INFO]', f"cache MISS for {concept} — generating with the LLM")
    # SPL: GENERATE write_section(@concept, @book_context) INTO @section
    _prompt = WRITE_SECTION_PROMPT.format(concept=concept, book_context=book_context)
    section = _llm_call(_prompt)
    print(f'@section:', section[:200] if isinstance(section, str) else section)
    # SPL: CALL check_terms(@section, @required_terms) INTO @check
    check = check_terms(section, required_terms)
    print(f'@check:', check)
    # SPL: EVALUATE @check
    if "fail" in str(check).lower():
        # SPL: LOGGING f'Coverage check failed: {@check} — refining' LEVEL INFO
        print('[INFO]', f"Coverage check failed: {check} — refining")
        # SPL: GENERATE refine_section(@section, @check) INTO @section
        _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback=check)
        section = _llm_call(_prompt)
        print(f'@section:', section[:200] if isinstance(section, str) else section)
    else:
        # SPL: LOGGING 'Coverage check passed' LEVEL INFO
        print('[INFO]', 'Coverage check passed')
    # SPL: CALL cache_put(@concept, @section) INTO @cache_key
    cache_key = cache_put(concept, section)
    print(f'@cache_key:', cache_key)
    # SPL: LOGGING f'Stored verified section — cache key {@cache_key}' LEVEL INFO
    print('[INFO]', f"Stored verified section — cache key {cache_key}")
    # SPL: CALL elapsed_ms(@t_section_start) INTO @section_ms
    section_ms = elapsed_ms(t_section_start)
    print(f'@section_ms:', section_ms)
    # SPL: LOGGING f'[timing] {@concept}: {@section_ms} ms (generated)' LEVEL INFO
    print('[INFO]', f"[timing] {concept}: {section_ms} ms (generated)")
else:
    # SPL: LOGGING f'cache HIT for {@concept} — reusing verified section (0 LLM calls)' LEVEL INFO
    print('[INFO]', f"cache HIT for {concept} — reusing verified section (0 LLM calls)")
    # SPL: CALL elapsed_ms(@t_section_start) INTO @section_ms
    section_ms = elapsed_ms(t_section_start)
    print(f'@section_ms:', section_ms)
    # SPL: LOGGING f'[timing] {@concept}: {@section_ms} ms (cached)' LEVEL INFO
    print('[INFO]', f"[timing] {concept}: {section_ms} ms (cached)")

In [ ]:
# SPL: CALL elapsed_ms(@t_workflow_start) INTO @total_ms
total_ms = elapsed_ms(t_workflow_start)
print(f'@total_ms:', total_ms)

In [ ]:
# SPL: LOGGING f'[timing] workflow total: {@total_ms} ms' LEVEL INFO
print('[INFO]', f"[timing] workflow total: {total_ms} ms")

In [ ]:
# SPL: COMMIT @section WITH status='complete'
import json
_output = section
_result_path = Path("chapter-1-vector-spaces-1A-real-complex_output.json")
_result_path.write_text(json.dumps(_output, indent=2, default=str))
print(f"Committed to {_result_path}")
# status: status='complete'